In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/online-retail-data-set-from-uci-ml-repo/Online Retail.xlsx
/kaggle/input/online-retail-ii-data-set-from-ml-repository/online_retail_II.xlsx


In [2]:
!pip install --upgrade pip

     |████████████████████████████████| 1.5 MB 1.2 MB/s 
  Attempting uninstall: pip
    Found existing installation: pip 21.0
    Uninstalling pip-21.0:
      Successfully uninstalled pip-21.0


##############################################################
# CLTV Prediction with Gamma-Gamma and BGNBD
##############################################################


# Updated:
# Quote: General comments are same 
# Some developments has been made.
# 1. Monetary simplified.
# 2. freq > 1 reduction has been made.
# 3. Recency personalized. (last purchase date - first purchase date)
# 4. Depending on the previous entry recency has been changed.
# 5. recency_weekly variable's name changed as "recency_weekly_cltv_p".
# 6. As a result general comments and course not changed.


# 4 Steps CLTV

# 1. Data Preperation
# 2. With BG-NBD Model,Expected Sale Forecasting values calculation.
# 3. With Gamma-Gamma Model,Expected Average Profit values calculation.
# 4. With BG-NBD and Gamma-Gamma models calculate specific timeframe's cltv values.


# Formulas:
# CLTV = (Customer_Value / Churn_Rate) x Profit_margin.
# Customer_Value = Average_Order_Value * Purchase_Frequency

##############################################################
# 1. Data Preperation
##############################################################

In [3]:
!pip install lifetimes

     |████████████████████████████████| 584 kB 1.3 MB/s 


In [4]:
import datetime as dt
import matplotlib.pyplot as plt
from lifetimes import BetaGeoFitter
from lifetimes import GammaGammaFitter
from lifetimes.plotting import plot_period_transactions
from sklearn.preprocessing import MinMaxScaler

In [5]:
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",None)
pd.set_option("display.float_format",lambda x: '%.5f'%x)

In [6]:
def outlier_thresholds(dataframe,variable):
    quartile1=dataframe[variable].quantile(0.01)
    quartile3=dataframe[variable].quantile(0.99)
    interquartile_range=quartile3-quartile1
    up_limit=quartile3 + 1.5 * interquartile_range
    low_limit=quartile1 - 1.5 * interquartile_range
    return low_limit,up_limit

In [7]:
def replace_with_thresholds(dataframe,variable):
    low_limit,up_limit = outlier_thresholds(dataframe,variable)
    #dataframe.loc[(dataframe[variable] < low_limit),variable]=low_limit
    dataframe.loc[(dataframe[variable]>up_limit),variable]=up_limit
    

In [8]:
!pip install openpyxl

     |████████████████████████████████| 243 kB 1.3 MB/s 
  Created wheel for et-xmlfile: filename=et_xmlfile-1.0.1-py3-none-any.whl size=8913 sha256=405163fc8bc5222e70a6e29d6fed2deb40dc473dc011c2ab7df6d105b030195e
  Stored in directory: /root/.cache/pip/wheels/e2/bd/55/048b4fd505716c4c298f42ee02dffd9496bb6d212b266c7f31
Successfully built et-xmlfile


In [9]:
df_=pd.read_excel("/kaggle/input/online-retail-ii-data-set-from-ml-repository/online_retail_II.xlsx",sheet_name="Year 2010-2011")

In [10]:
df=df_.copy()

In [11]:
df.shape
df.info()
df.head()
df.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[ns]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


,count,mean,std,min,25%,50%,75%,max
Quantity,541910.00000,9.55223,218.08096,-80995.00000,1.00000,3.00000,10.00000,80995.00000
Price,541910.00000,4.61114,96.75977,-11062.06000,1.25000,2.08000,4.13000,38970.00000
Customer ID,406830.00000,15287.68416,1713.60307,12346.00000,13953.00000,15152.00000,16791.00000,18287.00000


In [12]:
df.columns
df["Country"].values

array(['United Kingdom', 'United Kingdom', 'United Kingdom', ...,
       'France', 'France', 'France'], dtype=object)

In [13]:
df = df[(df["Country"] == "United Kingdom")]

In [14]:
df.dropna(inplace=True)

In [15]:
df=df[~df["Invoice"].str.contains("C",na=False)]

In [16]:
df=df[df["Quantity"]>0]

In [17]:
replace_with_thresholds(df,"Quantity")

In [18]:
replace_with_thresholds(df,"Price")

In [19]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Quantity,354345.00000,10.67687,22.07553,1.00000,2.00000,4.00000,12.00000,248.50000
Price,354345.00000,2.81504,2.92055,0.00000,1.25000,1.95000,3.75000,31.56000
Customer ID,354345.00000,15552.43622,1594.54603,12346.00000,14194.00000,15522.00000,16931.00000,18287.00000


In [20]:
df["TotalPrice"]=df["Quantity"]*df["Price"]

In [21]:
df["InvoiceDate"].max()

Timestamp('2011-12-09 12:49:00')

In [22]:
today_date=dt.datetime(2011,12,11)

#############################################
# RFM Table
#############################################

In [23]:
## dynamic recency per customer
rfm=df.groupby("Customer ID").agg({"InvoiceDate":
                                  [lambda date:(date.max()-date.min()).days,
                                   lambda date:(today_date - date.min()).days],
                                  "Invoice":lambda num:num.nunique(),
                                  "TotalPrice":lambda TotalPrice:TotalPrice.sum()})

In [24]:
rfm.columns=rfm.columns.droplevel(0)

In [25]:
## recency_cltv_p
rfm.columns=["recency_cltv_p","T","frequency","monetary"]

In [26]:
##simplyfied monetary_avg
rfm["monetary"]=rfm["monetary"]/rfm["frequency"]

In [27]:
rfm.rename(columns={"monetary":"monetary_avg"},inplace=True)

In [28]:
# for BGNBD calculation of WEEKLY RECENCY and WEEKLY T
## recency_weekly_p
rfm["recency_weekly_p"]=rfm["recency_cltv_p"]/7
rfm["T_weekly"]=rfm["T"]/7


In [29]:
##Check
rfm=rfm[rfm["monetary_avg"]>0]

In [30]:
##freq>1
rfm=rfm[(rfm["frequency"]>1)]
rfm["frequency"]=rfm["frequency"].astype(int)

In [31]:
##############################################################
# 2. BG/NBD Model
##############################################################

In [32]:
bgf=BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(rfm["frequency"],
       rfm["recency_weekly_p"],
       rfm["T_weekly"])

<lifetimes.BetaGeoFitter: fitted with 2570 subjects, a: 0.12, alpha: 11.66, b: 2.51, r: 2.21>

In [33]:
##############################################################
# 3. GAMMA-GAMMA Model
##############################################################


ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(rfm["frequency"],rfm["monetary_avg"])

<lifetimes.GammaGammaFitter: fitted with 2570 subjects, p: 3.82, q: 0.35, v: 3.75>

In [34]:
###Task 1
##Calculate CLTV for UK customers in 2010-2011
cltv = ggf.customer_lifetime_value(bgf,
                                   rfm['frequency'],
                                   rfm['recency_weekly_p'],
                                   rfm['T_weekly'],
                                   rfm['monetary_avg'],
                                   time=6,  # 6 months
                                   freq="W",  # T's frequency info.
                                   discount_rate=0.01)
cltv.head(20)

Customer ID
12747.00000    1937.04614
12748.00000   12365.79618
12749.00000    3446.01044
12820.00000     631.93933
12822.00000    1612.09665
12823.00000    1095.50846
12826.00000     757.01444
12827.00000    1009.14372
12828.00000    1125.11359
12829.00000      16.11081
12830.00000    4663.11033
12832.00000     765.80585
12836.00000    1749.83590
12838.00000     571.73922
12839.00000    2495.27206
12840.00000    1169.42997
12841.00000    1669.02324
12842.00000    1023.29302
12843.00000     991.50487
12844.00000     566.81490
Name: clv, dtype: float64

In [35]:
cltv=cltv.reset_index()

In [36]:
cltv.sort_values(by="clv",ascending=False)

,Customer ID,clv
2486,18102.00000,85651.01047
589,14096.00000,55650.64677
2184,17450.00000,48533.31011
2213,17511.00000,36797.00673
1804,16684.00000,25083.02541
406,13694.00000,25060.70871
587,14088.00000,25010.05913
1173,15311.00000,23591.38948
133,13089.00000,22927.69296
1057,15061.00000,21123.08206


In [37]:
rfm_cltv_1_task=rfm.merge(cltv,on="Customer ID",how="left")
scaler=MinMaxScaler(feature_range=(0,100))

In [38]:
scaled1=scaler.fit_transform(rfm_cltv_1_task[["clv"]])

In [39]:
scaled1

array([[ 2.26155667],
       [14.43742008],
       [ 4.02331556],
       ...,
       [ 0.37993369],
       [ 1.1480845 ],
       [ 2.23946858]])

In [40]:
rfm_cltv_1_task["scaled_cltv"]=scaled1

In [41]:
rfm_cltv_1_task[rfm_cltv_1_task["monetary_avg"] >= 1000].sort_values("monetary_avg",ascending=False).head(20)

,Customer ID,recency_cltv_p,T,frequency,monetary_avg,recency_weekly_p,T_weekly,clv,scaled_cltv
587,14088.00000,312,323,13,3859.60154,44.57143,46.14286,25010.05913,29.19996
2486,18102.00000,366,368,60,3584.88775,52.28571,52.57143,85651.01047,100.00000
589,14096.00000,97,102,17,3159.07706,13.85714,14.57143,55650.64677,64.97372
2213,17511.00000,370,374,31,2921.95194,52.85714,53.42857,36797.00673,42.96156
2184,17450.00000,359,368,46,2629.52989,51.28571,52.57143,48533.31011,56.66403
129,13081.00000,359,372,11,2575.62273,51.28571,53.14286,12971.29873,15.14436
1366,15749.00000,97,333,3,2521.44667,13.85714,47.57143,1867.61168,2.18049
1954,16984.00000,41,131,2,2240.67500,5.85714,18.71429,6119.76397,7.14500
1804,16684.00000,353,359,28,2120.04696,50.42857,51.28571,25083.02541,29.28515
1485,16000.00000,0,3,3,2055.78667,0.00000,0.42857,21110.80764,24.64747


In [42]:
rfm_cltv_1_task[rfm_cltv_1_task["recency_weekly_p"] <= 10].head(20)

,Customer ID,recency_cltv_p,T,frequency,monetary_avg,recency_weekly_p,T_weekly,clv,scaled_cltv
4,12822.00000,16,88,2,474.44000,2.28571,12.57143,1612.09665,1.88217
7,12827.00000,38,45,3,143.38333,5.42857,6.42857,1009.14372,1.17820
9,12829.00000,23,361,2,138.11000,3.28571,51.57143,16.11081,0.01881
11,12832.00000,65,99,2,191.51500,9.28571,14.14286,765.80585,0.89410
20,12845.00000,22,291,4,88.52250,3.14286,41.57143,4.34974,0.00508
24,12856.00000,55,64,6,363.32167,7.85714,9.14286,3431.50826,4.00638
29,12872.00000,38,366,2,299.98500,5.42857,52.28571,55.23347,0.06449
32,12878.00000,29,267,2,427.49500,4.14286,38.14286,186.38245,0.21761
33,12879.00000,47,92,3,191.07333,6.71429,13.14286,913.06575,1.06603
34,12882.00000,24,34,2,731.52000,3.42857,4.85714,4583.15620,5.35097


In [43]:
rfm_cltv_1_task.sort_values("scaled_cltv",ascending=False)

,Customer ID,recency_cltv_p,T,frequency,monetary_avg,recency_weekly_p,T_weekly,clv,scaled_cltv
2486,18102.00000,366,368,60,3584.88775,52.28571,52.57143,85651.01047,100.00000
589,14096.00000,97,102,17,3159.07706,13.85714,14.57143,55650.64677,64.97372
2184,17450.00000,359,368,46,2629.52989,51.28571,52.57143,48533.31011,56.66403
2213,17511.00000,370,374,31,2921.95194,52.85714,53.42857,36797.00673,42.96156
1804,16684.00000,353,359,28,2120.04696,50.42857,51.28571,25083.02541,29.28515
406,13694.00000,369,374,50,1267.36260,52.71429,53.42857,25060.70871,29.25909
587,14088.00000,312,323,13,3859.60154,44.57143,46.14286,25010.05913,29.19996
1173,15311.00000,373,374,91,667.59681,53.28571,53.42857,23591.38948,27.54362
133,13089.00000,366,370,97,605.18660,52.28571,52.85714,22927.69296,26.76874
1057,15061.00000,368,373,48,1108.30781,52.57143,53.28571,21123.08206,24.66180


In [44]:
##TASK 2
#Calculate cltv for UK customers for 1 and 12 months.
#Analyze highest cltv' for first 10 per month and 12 months is there any difference,and talk about 

In [45]:
cltv2_1month = ggf.customer_lifetime_value(bgf,
                                   rfm['frequency'],
                                   rfm['recency_weekly_p'],
                                   rfm['T_weekly'],
                                   rfm['monetary_avg'],
                                   time=1,  # 1 month
                                   freq="W",  # T's frequency info.
                                   discount_rate=0.01)

In [46]:
cltv2_12months=ggf.customer_lifetime_value(bgf,
                                   rfm['frequency'],
                                   rfm['recency_weekly_p'],
                                   rfm['T_weekly'],
                                   rfm['monetary_avg'],
                                   time=12,  # 12 months
                                   freq="W",  # T's frequency info.
                                   discount_rate=0.01)

In [47]:
cltv2_1month.head(10)

Customer ID
12747.00000    336.77883
12748.00000   2148.37567
12749.00000    604.07100
12820.00000    110.12485
12822.00000    286.92234
12823.00000    191.12371
12826.00000    131.66943
12827.00000    181.13361
12828.00000    198.69579
12829.00000      2.80551
Name: clv, dtype: float64

In [48]:
cltv2_12months.head(10)

Customer ID
12747.00000    3698.38114
12748.00000   23623.99689
12749.00000    6538.82768
12820.00000    1204.32622
12822.00000    3029.79454
12823.00000    2085.98885
12826.00000    1444.88055
12827.00000    1888.15775
12828.00000    2124.46518
12829.00000      30.72005
Name: clv, dtype: float64

In [49]:
cltv2_1month=cltv2_1month.reset_index()

In [50]:
cltv2_12months=cltv2_12months.reset_index()

In [51]:
cltv2_1month.sort_values(by="clv",ascending=False).head(20)

,Customer ID,clv
2486,18102.00000,14884.97500
589,14096.00000,9855.87989
2184,17450.00000,8434.76472
2213,17511.00000,6394.32432
1804,16684.00000,4361.05396
587,14088.00000,4355.48526
406,13694.00000,4354.46853
1173,15311.00000,4098.86945
133,13089.00000,3984.05150
1485,16000.00000,3843.97950


In [52]:
cltv2_12months.sort_values(by="clv",ascending=False).head(20)

,Customer ID,clv
2486,18102.00000,163591.12676
589,14096.00000,104900.44292
2184,17450.00000,92694.27482
2213,17511.00000,70285.65230
1804,16684.00000,47890.36421
406,13694.00000,47871.90000
587,14088.00000,47688.86365
1173,15311.00000,45067.80939
133,13089.00000,43795.47286
1057,15061.00000,40348.81669


In [53]:
rfm_cltv_2_task_1month=rfm.merge(cltv2_1month,on="Customer ID",how="left")

In [54]:
rfm_cltv_2_task_12months=rfm.merge(cltv2_12months,on="Customer ID",how="left")

In [55]:
scaled2_1month=scaler.fit_transform(rfm_cltv_2_task_1month[["clv"]])

In [56]:
scaled2_12months=scaler.fit_transform(rfm_cltv_2_task_12months[["clv"]])

In [57]:
rfm_cltv_2_task_1month["scaled_cltv"]=scaled2_1month

In [58]:
rfm_cltv_2_task_12months["scaled_cltv"]=scaled2_12months

In [59]:
rfm_cltv_2_task_1month.head(20)

,Customer ID,recency_cltv_p,T,frequency,monetary_avg,recency_weekly_p,T_weekly,clv,scaled_cltv
0,12747.00000,366,370,11,381.45545,52.28571,52.85714,336.77883,2.26254
1,12748.00000,372,374,210,153.82814,53.14286,53.42857,2148.37567,14.43318
2,12749.00000,209,214,5,814.48800,29.85714,30.57143,604.07100,4.05826
3,12820.00000,323,327,4,235.58500,46.14286,46.71429,110.12485,0.73984
4,12822.00000,16,88,2,474.44000,2.28571,12.57143,286.92234,1.92760
5,12823.00000,221,297,5,351.90000,31.57143,42.42857,191.12371,1.28400
6,12826.00000,362,366,7,210.67429,51.71429,52.28571,131.66943,0.88458
7,12827.00000,38,45,3,143.38333,5.42857,6.42857,181.13361,1.21689
8,12828.00000,127,131,6,169.78500,18.14286,18.71429,198.69579,1.33487
9,12829.00000,23,361,2,138.11000,3.28571,51.57143,2.80551,0.01885


In [60]:
rfm_cltv_2_task_12months.head(20)

,Customer ID,recency_cltv_p,T,frequency,monetary_avg,recency_weekly_p,T_weekly,clv,scaled_cltv
0,12747.00000,366,370,11,381.45545,52.28571,52.85714,3698.38114,2.26075
1,12748.00000,372,374,210,153.82814,53.14286,53.42857,23623.99689,14.44088
2,12749.00000,209,214,5,814.48800,29.85714,30.57143,6538.82768,3.99706
3,12820.00000,323,327,4,235.58500,46.14286,46.71429,1204.32622,0.73618
4,12822.00000,16,88,2,474.44000,2.28571,12.57143,3029.79454,1.85205
5,12823.00000,221,297,5,351.90000,31.57143,42.42857,2085.98885,1.27512
6,12826.00000,362,366,7,210.67429,51.71429,52.28571,1444.88055,0.88323
7,12827.00000,38,45,3,143.38333,5.42857,6.42857,1888.15775,1.15419
8,12828.00000,127,131,6,169.78500,18.14286,18.71429,2124.46518,1.29864
9,12829.00000,23,361,2,138.11000,3.28571,51.57143,30.72005,0.01878


# # As the estimated period increases, the difference in expected income increases. There is a directly proportional relationship between the two

1. According to 6-month CLTV for 2010-2011 UK customers, you can put all your customers into 3 groups (A,B,C) and add to the dataset  
- Analyze the 3 groups for other variables in the data set

In [61]:
rfm_cltv_3_task=rfm.merge(cltv,on="Customer ID",how="left")

In [62]:
rfm_cltv_3_task=ggf.customer_lifetime_value(bgf,
                                   rfm['frequency'],
                                   rfm['recency_weekly_p'],
                                   rfm['T_weekly'],
                                   rfm['monetary_avg'],
                                   time=6,  # 6months
                                   freq="W",  # T's frequency info.
                                   discount_rate=0.01)

In [63]:
rfm_cltv_3_task=rfm_cltv_3_task.reset_index()

In [64]:
rfm_cltv_3_task.sort_values(by="clv").head(20)

,Customer ID,clv
2380,17850.00000,0.00000
1079,15107.00000,0.80661
1829,16725.00000,1.12937
2407,17912.00000,1.84031
1437,15881.00000,2.65599
947,14813.00000,3.03037
1214,15422.00000,3.33221
20,12845.00000,4.34974
1515,16048.00000,4.73538
1862,16792.00000,6.39550


In [65]:
scaled3=scaler.fit_transform(rfm_cltv_3_task[["clv"]])

In [66]:
rfm_cltv_3_task["scaled_cltv"]=scaled3

In [67]:
task3_last=rfm_cltv_3_task.sort_values(by="scaled_cltv",ascending=False).head(20)

In [68]:
task3_last["segment"]=pd.qcut(gorev3_last["scaled_cltv"],3,labels=["C","B","A"])

NameError: name 'gorev3_last' is not defined

In [69]:
task3_last["scaled_cltv"].astype(int)

2486    99
589     64
2184    56
2213    42
1804    29
406     29
587     29
1173    27
133     26
1057    24
1485    24
692     24
1378    21
2499    20
2377    18
454     17
2150    16
2424    16
1503    15
139     15
Name: scaled_cltv, dtype: int64

In [70]:
task3_last.head(20)

,Customer ID,clv,scaled_cltv
2486,18102.00000,85651.01047,100.00000
589,14096.00000,55650.64677,64.97372
2184,17450.00000,48533.31011,56.66403
2213,17511.00000,36797.00673,42.96156
1804,16684.00000,25083.02541,29.28515
406,13694.00000,25060.70871,29.25909
587,14088.00000,25010.05913,29.19996
1173,15311.00000,23591.38948,27.54362
133,13089.00000,22927.69296,26.76874
1057,15061.00000,21123.08206,24.66180
